In [1]:
import ast
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

warnings.filterwarnings("ignore")

In [17]:
def load_data(path):
    df = pd.read_csv(path)
    print(f"✓ Loaded {len(df)} jobs, {df['final_category'].nunique()} categories")
    return df

In [3]:
print(df.columns)

Index(['title', 'company', 'location', 'date', 'job_link', 'description',
       'search_keyword', 'title_cleaned', 'cluster_id', 'auto_category',
       'final_category', 'relevant_section', 'extracted_skills_list',
       'extracted_skills', 'num_skills'],
      dtype='object')


,title,company,location,date,job_link,description,search_keyword,title_cleaned,cluster_id,auto_category,final_category,relevant_section,extracted_skills_list,extracted_skills,num_skills
0,BI / Data Engineer,KPMG Tunisia,"Tunis, Tunis, Tunisia",2/4/2026,https://tn.linkedin.com/jobs/view/bi-data-engi...,Vos Challenges\nL’Ingénieur BI / Data intervie...,data engineer,bi / data engineer,11.0,Engineer Software,Software Engineering,Vos Challenges\nL’Ingénieur BI / Data intervie...,"['analytics', 'ETL', 'dashboards', 'SQL']","analytics, ETL, dashboards, SQL",4
1,Data Engineer,Devoteam,"Tunis, Tunis, Tunisia",2/16/2026,https://tn.linkedin.com/jobs/view/data-enginee...,"Chez Devoteam, nous sommes des « Digital Trans...",data engineer,data engineer,11.0,Engineer Software,Software Engineering,compétences variées (consultants en transforma...,"['java', 'dataset', 'SQL', 'spring boot', 'sca...","java, dataset, SQL, spring boot, scala, CI/CD,...",7
2,Data Engineer [F/M/X],Mantu,"Tunis, Tunisia",2/6/2026,https://tn.linkedin.com/jobs/view/data-enginee...,Who are we?\nMantu is an independent internati...,data engineer,data engineer,11.0,Engineer Software,Software Engineering,mission: connecting and powering companies wit...,"['operational data', 'agile environment', 'dat...","operational data, agile environment, data pipe...",21
3,Data Analyst,Enovis,"Sfax, Sfax, Tunisia",1/29/2026,https://tn.linkedin.com/jobs/view/data-analyst...,Job Description\nPerforms statistical modeling...,data engineer,data analyst,8.0,Analyst Business,Data & Business Analysis,Job Description\nPerforms statistical modeling...,"['statistical', 'modeling', 'unstructured', 'd...","statistical, modeling, unstructured, datasets,...",7
4,Junior AI Engineer,Devoteam,"Tunis, Tunis, Tunisia",2/7/2026,https://tn.linkedin.com/jobs/view/junior-ai-en...,"Chez Devoteam, nous sommes des « Digital Trans...",data engineer,junior ai engineer,11.0,Engineer Software,Software Engineering,compétences variées (consultants en transforma...,"['NLP', 'Machine Learning', 'scikit - learn', ...","NLP, Machine Learning, scikit - learn, spacy, ...",9


In [5]:
def drop_unused_columns(df):
    cols_to_drop = ['description', 'auto_category', 'date']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    print(f"✓ Dropped unused columns: {cols_to_drop}")
    return df


In [6]:

def parse_skills_list(df):
    def safe_parse(val):
        if isinstance(val, list):
            return val
        try:
            parsed = ast.literal_eval(str(val))
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []

    df['skills_list'] = df['extracted_skills_list'].apply(safe_parse)

    # Clean individual skill tokens: remove BERT artefacts (##xxx), strip whitespace
    def clean_tokens(skills):
        cleaned = []
        for s in skills:
            s = str(s).strip()
            if s.startswith('##'):          # BERT subword artefact
                continue
            if len(s) < 2:                  # single chars
                continue
            if re.fullmatch(r'[^a-zA-Z0-9+#./\- ]+', s):  # pure punctuation
                continue
            cleaned.append(s.lower())
        return cleaned

    df['skills_list'] = df['skills_list'].apply(clean_tokens)
    df['num_skills']  = df['skills_list'].apply(len)
    print(f"✓ Parsed & cleaned skill lists")
    return df

In [7]:
def consolidate_categories(df, min_jobs=10):
    counts = df['final_category'].value_counts()
    rare   = counts[counts < min_jobs].index.tolist()
    df['domain'] = df['final_category'].apply(
        lambda x: 'Other' if x in rare else x
    )
    # Drop 'Other' rows — too noisy for recommendation
    before = len(df)
    df = df[df['domain'] != 'Other'].reset_index(drop=True)
    print(f"✓ Consolidated {len(rare)} rare categories → dropped {before - len(df)} rows")
    print(f"  Valid domains: {sorted(df['domain'].unique())}")
    return df

In [8]:
def normalize_location(df):
    def extract_city(loc):
        if pd.isna(loc):
            return 'unknown'
        loc = str(loc)
        # "Tunis, Tunis, Tunisia" → "tunis"
        city = loc.split(',')[0].strip().lower()
        return city

    df['city'] = df['location'].apply(extract_city)
    print(f"✓ Normalized location → city ({df['city'].nunique()} unique cities)")
    return df

In [9]:
def build_domain_profiles(df):
    """
    For each domain, aggregate all skills across all jobs.
    Returns:
        domain_profiles : dict { domain: Counter({skill: total_count}) }
        domain_top_skills: dict { domain: [(skill, count), ...] top 25 }
    """
    domain_profiles  = {}
    domain_top_skills = {}

    for domain, group in df.groupby('domain'):
        all_skills = []
        for skills in group['skills_list']:
            all_skills.extend(skills)
        counts = Counter(all_skills)
        domain_profiles[domain]   = counts
        domain_top_skills[domain] = counts.most_common(25)

    print(f"✓ Built skill profiles for {len(domain_profiles)} domains")
    return domain_profiles, domain_top_skills


In [10]:
def build_tfidf_matrices(df, domain_top_skills):
    """
    Two TF-IDF spaces:
      1. skill_vectorizer   → trained on job skill strings  (used to encode user)
      2. title_vectorizer   → trained on job title strings  (secondary signal)

    Domain profile matrix:
      Each domain = one document = its top skills repeated by relative frequency.
    """

    # --- Skills TF-IDF on individual jobs ---
    skill_strings = df['skills_list'].apply(lambda x: ' '.join(x))
    skill_vectorizer = TfidfVectorizer(
        analyzer    = 'word',
        token_pattern = r'[^\s]+',
        min_df      = 2,       # skill must appear in at least 2 jobs
        max_df      = 0.95,    # ignore skills in >95% of jobs (too generic)
        sublinear_tf = True,
    )
    job_skill_matrix = skill_vectorizer.fit_transform(skill_strings)

    # --- Title TF-IDF on individual jobs ---
    title_vectorizer = TfidfVectorizer(
        analyzer     = 'word',
        min_df       = 2,
        max_df       = 0.9,
        ngram_range  = (1, 2),
        sublinear_tf = True,
    )
    job_title_matrix = title_vectorizer.fit_transform(df['title_clean_text'])

    # --- Domain profile matrix (for domain ranking) ---
    domain_names    = list(domain_top_skills.keys())
    domain_docs     = []
    for domain in domain_names:
        top = domain_top_skills[domain]
        max_count = top[0][1] if top else 1
        tokens = []
        for skill, count in top:
            repeat = max(1, round(6 * count / max_count))
            tokens.extend([skill] * repeat)
        domain_docs.append(' '.join(tokens))

    domain_matrix = skill_vectorizer.transform(domain_docs)

    print(f"✓ TF-IDF matrices built")
    print(f"  Skills vocab size : {len(skill_vectorizer.vocabulary_)}")
    print(f"  Title vocab size  : {len(title_vectorizer.vocabulary_)}")
    print(f"  Job skill matrix  : {job_skill_matrix.shape}")

    return (skill_vectorizer, title_vectorizer,
            job_skill_matrix, job_title_matrix,
            domain_matrix, domain_names)


In [11]:
def rank_domains(user_skills, skill_vectorizer, domain_matrix, domain_names):
    """
    Encode user skills → cosine similarity against domain profiles.
    Returns sorted list of (domain, score).
    """
    user_text = ' '.join(user_skills)
    user_vec  = skill_vectorizer.transform([user_text])
    scores    = cosine_similarity(user_vec, domain_matrix).flatten()
    ranked    = sorted(zip(domain_names, scores), key=lambda x: x[1], reverse=True)
    return ranked


In [12]:
def get_missing_skills(user_skills, domain, domain_top_skills, top_n=6):
    """
    Top skills in the domain the user lacks, ordered by domain importance.
    """
    user_lower = {s.lower() for s in user_skills}
    top        = domain_top_skills.get(domain, [])
    max_count  = top[0][1] if top else 1

    missing = []
    for skill, count in top:
        if skill not in user_lower:
            importance = count / max_count  # 0–1
            missing.append((skill, importance))

    return missing[:top_n]


In [13]:
def find_best_job(user_skills, domain, df, skill_vectorizer,
                  job_skill_matrix, job_title_matrix, title_vectorizer,
                  skill_weight=0.75, title_weight=0.25):
    """
    Hybrid similarity: skill cosine (75%) + title cosine (25%).
    Only considers jobs in the target domain.
    """
    domain_mask = df['domain'] == domain
    domain_idx  = df.index[domain_mask].tolist()

    if not domain_idx:
        return None, 0.0

    # Encode user
    user_skill_vec = skill_vectorizer.transform([' '.join(user_skills)])
    user_title_vec = title_vectorizer.transform([' '.join(user_skills)])

    # Subset matrices to domain jobs
    d_skill_mat = job_skill_matrix[domain_idx]
    d_title_mat = job_title_matrix[domain_idx]

    skill_sims = cosine_similarity(user_skill_vec, d_skill_mat).flatten()
    title_sims = cosine_similarity(user_title_vec, d_title_mat).flatten()

    # Combined score
    combined = skill_weight * skill_sims + title_weight * title_sims

    best_local_idx = int(np.argmax(combined))
    best_global_idx = domain_idx[best_local_idx]
    best_score = combined[best_local_idx]

    return df.iloc[best_global_idx], best_score

In [14]:
def get_user_skills():
    print("\n" + "═"*60)
    print("        CAREER DOMAIN RECOMMENDER")
    print("═"*60)
    print("  Enter your skills one by one.")
    print("  Press Enter after each skill.")
    print("  Type 'done' when finished.")
    print("─"*60)

    user_skills = []
    while True:
        raw = input("  Skill: ").strip()
        if raw.lower() == 'done':
            break
        if not raw:
            continue
        skill = raw.lower()
        if skill in user_skills:
            print(f"    ⚠  Already added.")
        else:
            user_skills.append(skill)
            print(f"    ✓  Added: {raw}")

    return user_skills

In [15]:
def get_user_skills():
    print("\n" + "═"*60)
    print("        CAREER DOMAIN RECOMMENDER")
    print("═"*60)
    print("  Enter your skills one by one.")
    print("  Press Enter after each skill.")
    print("  Type 'done' when finished.")
    print("─"*60)

    user_skills = []
    while True:
        raw = input("  Skill: ").strip()
        if raw.lower() == 'done':
            break
        if not raw:
            continue
        skill = raw.lower()
        if skill in user_skills:
            print(f"    ⚠  Already added.")
        else:
            user_skills.append(skill)
            print(f"    ✓  Added: {raw}")

    return user_skills


def display_results(user_skills, ranked_domains, domain_top_skills,
                    df, skill_vectorizer, job_skill_matrix,
                    job_title_matrix, title_vectorizer):

    if not user_skills:
        print("\n  No skills entered. Exiting.\n")
        return

    print(f"\n  Your skills ({len(user_skills)}): {', '.join(s.title() for s in user_skills)}")

    # ── Domain ranking ────────────────────────────────────────
    print("\n" + "═"*60)
    print("  DOMAIN RANKING")
    print("─"*60)
    for i, (domain, score) in enumerate(ranked_domains, 1):
        bar    = "█" * int(score * 28)
        marker = " ◀ BEST FIT" if i == 1 else ""
        print(f"  {i:2}. {domain:<35} {score:5.1%}  {bar}{marker}")

    best_domain, best_score = ranked_domains[0]

    # ── Best domain details ───────────────────────────────────
    print("\n" + "═"*60)
    print(f"  ✅  RECOMMENDED DOMAIN:  {best_domain}")
    print(f"      Match score:  {best_score:.1%}")
    print("─"*60)

    # Matched skills
    domain_skill_set = {s for s, _ in domain_top_skills.get(best_domain, [])}
    matched = [s for s in user_skills if s in domain_skill_set]

    print(f"\n  Skills you already have in this domain:")
    if matched:
        for s in matched:
            print(f"    ✓  {s.title()}")
    else:
        print("    (none matched directly — transferable skills may apply)")

    # Missing skills
    missing = get_missing_skills(user_skills, best_domain, domain_top_skills)

    print(f"\n  Skills to learn to excel in  {best_domain}:")
    stars = {0.7: "★★★  critical", 0.4: "★★☆  important", 0.0: "★☆☆  useful"}
    for skill, imp in missing:
        level = next(label for threshold, label in stars.items() if imp >= threshold)
        print(f"    →  {skill.title():<28}  {level}")

    # ── Job suggestion ────────────────────────────────────────
    print("\n" + "─"*60)
    print("  Finding best matching job post...")
    job, job_score = find_best_job(
        user_skills, best_domain, df,
        skill_vectorizer, job_skill_matrix,
        job_title_matrix, title_vectorizer
    )

    print("\n" + "═"*60)
    print("  💼  SUGGESTED JOB TO APPLY FOR")
    print("─"*60)

    if job is None:
        print("  No jobs found for this domain.")
    else:
        print(f"  Title    :  {job.get('title', 'N/A')}")
        print(f"  Company  :  {job.get('company', 'N/A')}")
        print(f"  Location :  {job.get('location', 'N/A')}")
        print(f"  Fit score:  {job_score:.1%}")

        # Skills overlap breakdown
        job_skills  = job.get('skills_list', [])
        if isinstance(job_skills, str):
            try:    job_skills = ast.literal_eval(job_skills)
            except: job_skills = []

        user_set    = set(user_skills)
        overlap     = [s for s in job_skills if s in user_set]
        gaps        = [s for s in job_skills if s not in user_set][:5]

        if overlap:
            print(f"\n  Your matching skills :  {', '.join(s.title() for s in overlap)}")
        if gaps:
            print(f"  Skills to work on    :  {', '.join(s.title() for s in gaps)}")

        # Link
        link_col = next((c for c in ['job_link', 'url', 'link', 'apply_url']
                         if c in job.index and pd.notna(job[c])), None)
        if link_col:
            print(f"\n  Apply here:  {job[link_col]}")

    print("\n" + "═"*60 + "\n")

    # ── Ask if user wants another domain explored ─────────────
    while True:
        ans = input("  Want details for another domain? (enter number or 'no'): ").strip()
        if ans.lower() in ('no', 'n', ''):
            break
        try:
            idx = int(ans) - 1
            if 0 <= idx < len(ranked_domains):
                alt_domain, alt_score = ranked_domains[idx]
                missing_alt = get_missing_skills(user_skills, alt_domain, domain_top_skills)
                job_alt, job_alt_score = find_best_job(
                    user_skills, alt_domain, df,
                    skill_vectorizer, job_skill_matrix,
                    job_title_matrix, title_vectorizer
                )
                print(f"\n  ── {alt_domain}  (match: {alt_score:.1%}) ──")
                print(f"  Skills to learn:")
                for skill, imp in missing_alt:
                    level = next(label for threshold, label in stars.items() if imp >= threshold)
                    print(f"    →  {skill.title():<28}  {level}")
                if job_alt is not None:
                    print(f"\n  Best job:  {job_alt.get('title', 'N/A')} @ {job_alt.get('company', 'N/A')}")
                    print(f"  Fit score: {job_alt_score:.1%}")
                    link_col = next((c for c in ['job_link', 'url', 'link']
                                     if c in job_alt.index and pd.notna(job_alt[c])), None)
                    if link_col:
                        print(f"  Apply:     {job_alt[link_col]}")
                print()
            else:
                print("  Invalid number.")
        except ValueError:
            print("  Please enter a number or 'no'.")

In [19]:
def build_pipeline(csv_path):
    print("\n" + "─"*60)
    print("  Building recommendation engine...")
    print("─"*60)

    # Load & engineer features
    df = load_data(csv_path)
    df = drop_unused_columns(df)
    df = parse_skills_list(df)
    df = consolidate_categories(df, min_jobs=10)
    df = normalize_location(df)
    df['title_clean_text'] = df['title_cleaned'].astype(str)

    # Build profiles & matrices
    domain_profiles, domain_top_skills = build_domain_profiles(df)
    (skill_vectorizer, title_vectorizer,
     job_skill_matrix, job_title_matrix,
     domain_matrix, domain_names) = build_tfidf_matrices(df, domain_top_skills)

    print("  Engine ready ✓\n")

    return (df, domain_top_skills,
            skill_vectorizer, title_vectorizer,
            job_skill_matrix, job_title_matrix,
            domain_matrix, domain_names)

In [20]:
def main():
    CSV_PATH = "/content/linkedin_jobs_with_extracted_skills_per_job4.csv"

    # Build once, run multiple times
    (df, domain_top_skills,
     skill_vectorizer, title_vectorizer,
     job_skill_matrix, job_title_matrix,
     domain_matrix, domain_names) = build_pipeline(CSV_PATH)

    while True:
        # Get user input
        user_skills = get_user_skills()

        if not user_skills:
            print("  No skills entered.")
        else:
            # Rank domains
            ranked = rank_domains(user_skills, skill_vectorizer, domain_matrix, domain_names)

            # Display full results
            display_results(
                user_skills, ranked, domain_top_skills,
                df, skill_vectorizer, job_skill_matrix,
                job_title_matrix, title_vectorizer
            )

        again = input("  Start over with new skills? (yes/no): ").strip().lower()
        if again not in ('yes', 'y'):
            print("\n  Goodbye!\n")
            break


if __name__ == "__main__":
    main()



────────────────────────────────────────────────────────────
  Building recommendation engine...
────────────────────────────────────────────────────────────
✓ Loaded 503 jobs, 27 categories
✓ Dropped unused columns: ['description', 'auto_category', 'date']
✓ Parsed & cleaned skill lists
✓ Consolidated 19 rare categories → dropped 82 rows
  Valid domains: ['Data & Business Analysis', 'Data Science & ML', 'Design & Graphics', 'DevOps Engineering', 'IT Consulting & ERP', 'IT Support & Technical Services', 'Product & Project Management', 'Software Engineering']
✓ Normalized location → city (22 unique cities)
✓ Built skill profiles for 8 domains
✓ TF-IDF matrices built
  Skills vocab size : 387
  Title vocab size  : 424
  Job skill matrix  : (421, 387)
  Engine ready ✓


════════════════════════════════════════════════════════════
        CAREER DOMAIN RECOMMENDER
════════════════════════════════════════════════════════════
  Enter your skills one by one.
  Press Enter after each skill.
 